In [2]:
import pandas as pd
import os
from itertools import permutations, combinations

## Unión de clorofila y reflectancias

In [3]:
# Cargamos los csv de los tifs
path = "saved_files/"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tiffs_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_tifs[nombre_sin_extension] = pd.read_csv(ruta_completa)

# Y de las boyas
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_boyas[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [4]:
band_names = {
    "Band_1": "rhow_B1",
    "Band_2": "rhow_B2",
    "Band_3": "rhow_B3",
    "Band_4": "rhow_B4",
    "Band_5": "rhow_B5",
    "Band_6": "rhow_B6",
    "Band_7": "rhow_B7",
    "Band_8": "rhow_B8",
    "Band_9": "rhown_B1",
    "Band_10": "rhown_B2",
    "Band_11": "rhown_B3",
    "Band_12": "rhown_B4",
    "Band_13": "rhown_B5",
    "Band_14": "rhown_B6",
}
for nombre_df, df in dfs_tifs.items():
    dfs_tifs[nombre_df] = df.rename(columns=band_names)


In [5]:
dfs_tifs["df_tiffs_c2x-complex-nets_1x1"].head(3)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,rhow_B7,rhow_B8,rhown_B1,rhown_B2,rhown_B3,rhown_B4,rhown_B5,rhown_B6,Band_15,Band_16
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.002320,0.002351,0.000930,0.023582,0.034339,0.043720,0.015355,0.009758,0.002110,0.001174,-8.000000e-45
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.001000,0.001059,0.000427,0.014022,0.022509,0.026334,0.007046,0.004299,0.000942,0.000236,-8.000000e-45
2,2017-06-30,CTD3,4181698,695238,0.017592,0.024643,0.031816,0.010574,0.007355,0.001946,0.001922,0.000778,0.017462,0.024656,0.031839,0.010682,0.007440,0.001862,0.104799,-8.000000e-45


In [6]:
dfs_boyas["df_boyas_upct_depth_gt_1"].head(3)

,Date,Buoy,Chl
0,2017-05-19,CTD1,0.883
1,2017-05-19,CTD10,0.837
2,2017-05-19,CTD11,0.899


In [7]:
merge_dict = {}

for df_tif_name, df_tif in dfs_tifs.items():
    for df_boya_name, df_boya in dfs_boyas.items():
        print(df_tif_name[9:], df_boya_name[9:])
        merge_dict[f"{df_tif_name[9:]}_{df_boya_name[9:]}"] = df_tif.merge(df_boya, how="inner", on=["Date", "Buoy"])    

c2x-nets_5x5 upct_depth_gt_1
c2x-nets_5x5 imida_depth_gt_1
c2x-nets_5x5 imida_depth_lt_2
c2x-nets_5x5 upct_depth_lt_2
c2x-nets_5x5 upct_depth_lt_1
c2x-nets_5x5 imida_depth_lt_1
c2x-nets_3x3 upct_depth_gt_1
c2x-nets_3x3 imida_depth_gt_1
c2x-nets_3x3 imida_depth_lt_2
c2x-nets_3x3 upct_depth_lt_2
c2x-nets_3x3 upct_depth_lt_1
c2x-nets_3x3 imida_depth_lt_1
c2x-complex-nets_5x5 upct_depth_gt_1
c2x-complex-nets_5x5 imida_depth_gt_1
c2x-complex-nets_5x5 imida_depth_lt_2
c2x-complex-nets_5x5 upct_depth_lt_2
c2x-complex-nets_5x5 upct_depth_lt_1
c2x-complex-nets_5x5 imida_depth_lt_1
c2x-complex-nets_1x1 upct_depth_gt_1
c2x-complex-nets_1x1 imida_depth_gt_1
c2x-complex-nets_1x1 imida_depth_lt_2
c2x-complex-nets_1x1 upct_depth_lt_2
c2x-complex-nets_1x1 upct_depth_lt_1
c2x-complex-nets_1x1 imida_depth_lt_1
c2x-complex-nets_9x9 upct_depth_gt_1
c2x-complex-nets_9x9 imida_depth_gt_1
c2x-complex-nets_9x9 imida_depth_lt_2
c2x-complex-nets_9x9 upct_depth_lt_2
c2x-complex-nets_9x9 upct_depth_lt_1
c2x-compl

In [27]:
for df_name, df in merge_dict.items():
    df.to_csv(f"saved_files/dataset/{df_name}.csv", index=False)

## Creación de features

In [10]:
# Cargamos los csv de los tifs
path = "saved_files/dataset/"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)

Fórmulas a utilizar:
- Diferencia normalizada $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_1) + R(\lambda_2)}$$
- Dall-Gitelson $$\left(\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}\right) \times R(\lambda_3)$$
- Diferencia normalizada 4 bandas $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_3) + R(\lambda_4)}$$
- Diferencia inversas $$\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}$$
- Diferencia relación 4 bands $$\frac{R(\lambda_1)}{R(\lambda_2)}-\frac{R(\lambda_3)}{R(\lambda_4)}$$

In [50]:
def diferencia_normalizada(band1, band2):
    value = (band1 - band2)/(band1 + band2)
    return value.round(3)

def dall_gitelson(band1, band2, band3):
    value = (1/(band1) - 1/(band2))*(band3)
    return value.round(3)

def diferencia_normalizada_4bandas(band1, band2, band3, band4):
    value = (band1 - band2)/(band3 + band4)
    return value.round(3)

def diferencia_inversas(band1, band2):
    value = 1/(band1) - 1/(band2)
    return value.round(3)

def diferencia_relacion_4bandas(band1, band2, band3, band4):
    value = band1/band2 - band3/band4
    return value.round(3)

Añadimos la diferencia normalizada y diferencia de inversas para las bandas de Ultra Blue, Blue, Green , Red, NIR1; lo hacemos dos veces, con rhow y con rhown.

In [51]:
index_list = []
def add_two_band_difs(data, bands):
    for i, band1 in enumerate(bands):
        for band2 in bands[i+1:]:
            colname_dif_norm = f"dif_norm_{band1}_{band2}"
            data[colname_dif_norm] = diferencia_normalizada(data[band1], data[band2])
            index_list.append(colname_dif_norm)
            colname_dif_inv = f"dif_inv_{band1}_{band2}"
            data[colname_dif_inv] = diferencia_inversas(data[band1], data[band2])
            index_list.append(colname_dif_inv)
    return data


Añadimos la relación de Dall-Gitelson, evitando repeticiones por la simetría de $f(b1,b2,b3)=−f(b2,b1,b3)$

In [52]:
index_dall_gitelson_list = []
def add_dall_gitelson(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita repeticiones de pares
        for band3 in bands:
            if band3 not in (band1, band2):  # evitar que band3 sea igual a los anteriores
                colname = f"dall_gitelson_{band1}_{band2}_{band3}"
                data[colname] = dall_gitelson(data[band1], data[band2], data[band3])
                index_dall_gitelson_list.append(colname)
    return data


Añadimos el índice de tipo diferencia normalizada que utiliza 4 bands, forzando a que estas 4 sean diferentes y evitando redundancia por las simetrías $(b1−b2)/(b3+b4)=(b1−b2)/(b4+b3)$ y $(b1−b2)/(b3+b4)=−(b2−b1)/(b3+b4)$

In [53]:
index_dif_norm_4bands_list = []
def add_norm_dif_4bands(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita invertir band1 y band2
        for band3, band4 in combinations(bands, 2):  # evita invertir band3 y band4
            # Asegurar que todas las bandas son distintas
            if len({band1, band2, band3, band4}) == 4:
                colname = f"index_4_bands_{band1}_{band2}_{band3}_{band4}"
                data[colname] = diferencia_normalizada_4bandas(
                    data[band1], data[band2], data[band3], data[band4]
                )
                index_dif_norm_4bands_list.append(colname)
    return data

Añadimos el cociente entre dos parejas de bandas, también evitando simetría, en este caso:

$\frac{b1}{b2} - \frac{b3}{b4} = -\left(\frac{b3}{b4} - \frac{b1}{b2} \right)$

In [55]:
index_dif_rel_4bands_list = []

def add_index_dif_rel_4bands(data, bands):
    for band1, band2, band3, band4 in permutations(bands, 4):
        # Evitar redundancias por simetría de términos
        # Criterio: solo aceptamos combinaciones donde el primer término es "menor" que el segundo
        if (band1, band2) < (band3, band4):  # evita generar la versión espejo con signo opuesto
            colname = f"index_diff_ratio_{band1}_{band2}_{band3}_{band4}"
            data[colname] = diferencia_relacion_4bandas(
                data[band1], data[band2], data[band3], data[band4]
            )
            index_dif_rel_4bands_list.append(colname)
    return data

Aplicamos todas las fórmulas anteriores sobre los dataframes del diccionario de dataframes, para los conjuntos de bandas de rhow y rhown.

In [60]:
band_sets = [
    ['rhow_B1', 'rhow_B2', 'rhow_B3', 'rhow_B4', 'rhow_B5'],
    ['rhown_B1', 'rhown_B2', 'rhown_B3', 'rhown_B4', 'rhown_B5']
]
for nombre_df, df in dfs.items():
    for bands_to_use in band_sets:
        df = add_two_band_difs(df, bands_to_use)
        df = add_dall_gitelson(df, bands_to_use)
        df = add_norm_dif_4bands(df, bands_to_use)
        df = add_index_dif_rel_4bands(df, bands_to_use)
    dfs[nombre_df] = df 

/tmp/ipykernel_7661/2634721933.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[colname] = diferencia_relacion_4bandas(
/tmp/ipykernel_7661/2634721933.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[colname] = diferencia_relacion_4bandas(
/tmp/ipykernel_7661/2634721933.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented f

Ya tenemos todos los dataframes con todas las columnas.

In [73]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1"].head(2)

,Date,Buoy,Latitude,Longitude,_rhow_B1,_rhow_B2,_rhow_B3,_rhow_B4,_rhow_B5,_rhow_B6,...,index_diff_ratio_rhown_B3_B4_B5_B1,index_diff_ratio_rhown_B3_B4_B5_B2,index_diff_ratio_rhown_B3_B5_B4_B1,index_diff_ratio_rhown_B3_B5_B4_B2,index_diff_ratio_rhown_B4_B1_B5_B2,index_diff_ratio_rhown_B4_B1_B5_B3,index_diff_ratio_rhown_B4_B2_B5_B1,index_diff_ratio_rhown_B4_B2_B5_B3,index_diff_ratio_rhown_B4_B3_B5_B1,index_diff_ratio_rhown_B4_B3_B5_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Renombramos columnas para que los nombres no incluyan varias veces rhow / rhown. Es decir, que index_diff_ratio_rhow_B1_rhow_B4_rhow_B2_rhow_B5 sea solamente index_diff_ratio_rhow_B1_B4_B2_B5.

In [66]:
import re

def compactar_prefijos_columnas(df):
    nuevo_nombre_columnas = {}

    for col in df.columns:
        # Detectar columnas con patrones tipo index_algo_rhow_B1_rhow_B2_...
        if re.search(r'(rhow|rhown)(_B\d+)+', col):
            partes = col.split('_')
            base = []
            bandas = []
            prefijo = None

            for parte in partes:
                if parte in ['rhow', 'rhown']:
                    if not prefijo:
                        prefijo = parte
                elif parte.startswith('B'):
                    bandas.append(parte)
                else:
                    base.append(parte)

            if prefijo and bandas:
                nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                nuevo_nombre_columnas[col] = nuevo_nombre

    # Renombrar columnas
    df = df.rename(columns=nuevo_nombre_columnas)
    return df


In [67]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = compactar_prefijos_columnas(df)

In [75]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1"].head(2)

,Date,Buoy,Latitude,Longitude,_rhow_B1,_rhow_B2,_rhow_B3,_rhow_B4,_rhow_B5,_rhow_B6,...,index_diff_ratio_rhown_B3_B4_B5_B1,index_diff_ratio_rhown_B3_B4_B5_B2,index_diff_ratio_rhown_B3_B5_B4_B1,index_diff_ratio_rhown_B3_B5_B4_B2,index_diff_ratio_rhown_B4_B1_B5_B2,index_diff_ratio_rhown_B4_B1_B5_B3,index_diff_ratio_rhown_B4_B2_B5_B1,index_diff_ratio_rhown_B4_B2_B5_B3,index_diff_ratio_rhown_B4_B3_B5_B1,index_diff_ratio_rhown_B4_B3_B5_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Guardamos los dataframes como csvs, con el mismo nombre que tenían pero añadiendo "_features" al final.

In [70]:
for df_name, df in dfs.items():
    df.to_csv(f"saved_files/dataset/{df_name}_features.csv", index=False)

['Date',
 'Buoy',
 'Latitude',
 'Longitude',
 '_rhow_B1',
 '_rhow_B2',
 '_rhow_B3',
 '_rhow_B4',
 '_rhow_B5',
 '_rhow_B6',
 '_rhow_B7',
 '_rhow_B8',
 '_rhown_B1',
 '_rhown_B2',
 '_rhown_B3',
 '_rhown_B4',
 '_rhown_B5',
 '_rhown_B6',
 'Band_15',
 'Band_16',
 'Chl',
 'dif_norm_rhow_B1_B2',
 'dif_inv_rhow_B1_B2',
 'dif_norm_rhow_B1_B3',
 'dif_inv_rhow_B1_B3',
 'dif_norm_rhow_B1_B4',
 'dif_inv_rhow_B1_B4',
 'dif_norm_rhow_B1_B5',
 'dif_inv_rhow_B1_B5',
 'dif_norm_rhow_B2_B3',
 'dif_inv_rhow_B2_B3',
 'dif_norm_rhow_B2_B4',
 'dif_inv_rhow_B2_B4',
 'dif_norm_rhow_B2_B5',
 'dif_inv_rhow_B2_B5',
 'dif_norm_rhow_B3_B4',
 'dif_inv_rhow_B3_B4',
 'dif_norm_rhow_B3_B5',
 'dif_inv_rhow_B3_B5',
 'dif_norm_rhow_B4_B5',
 'dif_inv_rhow_B4_B5',
 'dall_gitelson_rhow_B1_B2_B3',
 'dall_gitelson_rhow_B1_B2_B4',
 'dall_gitelson_rhow_B1_B2_B5',
 'dall_gitelson_rhow_B1_B3_B2',
 'dall_gitelson_rhow_B1_B3_B4',
 'dall_gitelson_rhow_B1_B3_B5',
 'dall_gitelson_rhow_B1_B4_B2',
 'dall_gitelson_rhow_B1_B4_B3',
 'dall_gi